# Pharmaceutical Pseudonymisation — Portfolio Demonstration

Uses synthetic pharmacy data. No NHS affiliation or real patient records.

**Security:** Set `PHARMACY_HMAC_KEY` via an approved secret-management mechanism before execution; do not commit keys. In production, use managed secret retrieval (for example, Azure Key Vault), restricted access, and approved governance.

**Scope:** Fact Sales audit logging is implemented in code; Dimension audit logging is not. Identifier mappings use driver-side `collect()` and are POC-scale only. Pseudonymised identifiers are not anonymous data.


In [ ]:
# 1. PSEUDONYMISATION SETUP

import hmac
import hashlib
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType

# TEST ONLY - production secret will come from Azure Key Vault
import os
SECRET_KEY = os.environ.get("PHARMACY_HMAC_KEY")
if not SECRET_KEY:
    raise RuntimeError("Set PHARMACY_HMAC_KEY securely before running this notebook")

def pseudonymise_id(value):
    if value is None:
        return None

    # Standardise the identifier before pseudonymising
    cleaned = str(value).replace(" ", "").replace("-", "").strip()

    return hmac.new(
        SECRET_KEY.encode("utf-8"),
        cleaned.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

# Make the function available to Spark
pseudonymise_udf = udf(pseudonymise_id, StringType())

In [ ]:
# PSEUDONYMISE PHARMACY DIM_PHARMACY (Including standalone and embedded PharmacyIDs across all string columns)

from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType
import re

# 1. Reading my original table
df_pharmacy = spark.read.table("dbo.PharmacyDimPharmacy")

# 2. Create a mapping of original IDs to HMAC value and Reuse existing HMAC UDF for consistency
df_id_mapping = (
    df_pharmacy
    .select("PharmacyID")
    .where(col("PharmacyID").isNotNull())
    .distinct()
    .withColumn("PseudoID", pseudonymise_udf(col("PharmacyID")))
)

# Convert the mapping into a Python dictionary
id_mapping = {
    row["PharmacyID"]: row["PseudoID"]
    for row in df_id_mapping.collect()
}

# 3. Create pattern to detect IDs anywhere within text, Longest IDs first to avoid overlapping matches
if id_mapping:

    pattern = re.compile(
        "|".join(
            re.escape(k)
            for k in sorted(id_mapping, key=len, reverse=True)
        )
    )

    def replace_pharmacy_ids(value):
        if value is None:
            return None

        return pattern.sub(
            lambda match: id_mapping[match.group(0)],
            value
        )

    replace_ids_udf = udf(replace_pharmacy_ids, StringType())

    # 4. Scan every string column
    df_pharmacy_curated = df_pharmacy


    for field in df_pharmacy.schema.fields:
        if isinstance(field.dataType, StringType):
            df_pharmacy_curated = df_pharmacy_curated.withColumn(
                field.name,
                replace_ids_udf(col(field.name))
            )

else:
    df_pharmacy_curated = df_pharmacy

# 5. Save curated table
df_pharmacy_curated.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("psy.PharmacyDimPharmacy_Curated")

# 6. Display results
display(df_pharmacy_curated)

In [ ]:
# PSEUDONYMISE PHARMACY FACT_SALES (Including standalone and embedded PharmacyIDs across all string columns)

from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType
import re

# 1. Reading my original Fact Sales table
df_sales = spark.read.table("dbo.PharmacyFactSales")

# 2. Create a mapping of original IDs to HMAC value and Reuse existing HMAC UDF for consistency
df_id_mapping = (
    df_sales
    .select("PharmacyID")
    .where(col("PharmacyID").isNotNull())
    .distinct()
    .withColumn("PseudoID", pseudonymise_udf(col("PharmacyID")))
)

# Convert the mapping into a Python dictionary
id_mapping = {
    row["PharmacyID"]: row["PseudoID"]
    for row in df_id_mapping.collect()
}

# 3. Create pattern to detect IDs anywhere within text, Longest IDs first to avoid overlapping matches
if id_mapping:

    pattern = re.compile(
        "|".join(
            re.escape(k)
            for k in sorted(id_mapping, key=len, reverse=True)
        )
    )

    def replace_pharmacy_ids(value):
        if value is None:
            return None

        return pattern.sub(
            lambda match: id_mapping[match.group(0)],
            value
        )

    replace_ids_udf = udf(replace_pharmacy_ids, StringType())

    # 4. Scan every string column
    df_sales_curated = df_sales

    for field in df_sales.schema.fields:
        if isinstance(field.dataType, StringType):
            df_sales_curated = df_sales_curated.withColumn(
                field.name,
                replace_ids_udf(col(field.name))
            )

else:
    df_sales_curated = df_sales

# 5. Save curated table
df_sales_curated.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("psy.PharmacyFactSales_Curated")

# 6. PROCESSING AUDIT LOG - FACT SALES

from pyspark.sql import Row
from pyspark.sql.types import StringType
from datetime import datetime, timezone
import uuid

# Unique ID and timestamp for this processing run
run_id = str(uuid.uuid4())
processed_at = datetime.now(timezone.utc)

log_rows = []

# Scan each original string column and count actual ID occurrences
for field in df_sales.schema.fields:

    column_name = field.name
    replacement_count = 0

    if isinstance(field.dataType, StringType) and id_mapping:

        # Count every matching PharmacyID occurrence
        values = df_sales.select(column_name).toLocalIterator()

        for row in values:
            value = row[column_name]

            if value is not None:
                replacement_count += len(
                    pattern.findall(value)
                )

    log_rows.append(
        Row(
            RunID=run_id,
            SourceTable="dbo.PharmacyFactSales",
            ColumnName=column_name,
            ReplacementsCount=replacement_count,
            ProcessedAt=processed_at
        )
    )

# 7. Create audit DataFrame
df_processing_log = spark.createDataFrame(log_rows)

# 8. Append log records rather than overwriting previous runs
df_processing_log.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("metadata.PseudonymisationProcessingLog")

# 9. Display processing summary
display(df_processing_log)

# 10. Display curated data
display(df_sales_curated)



In [ ]:
# VALIDATE THAT THE PSEUDONYMISED KEY STILL JOINS AS A RELATIONSHIP

validation = (
    df_sales_curated
    .select("PharmacyID")
    .distinct()
    .join(
        df_pharmacy_curated.select("PharmacyID").distinct(),
        on="PharmacyID",
        how="inner"
    )
)

print("Matching pseudonymised PharmacyIDs across both tables:")
print(validation.count())

display(validation)